In [1]:
# 1. Check the GPU: all team runs use the same GPU type (L4)
import torch
gpu = torch.cuda.get_device_properties(0)
print(gpu.name, f"{gpu.total_memory / 2**30:.1f} GiB")
assert "L4" in gpu.name, "Switch the runtime to an L4 GPU (Runtime > Change runtime type)"

NVIDIA L4 22.0 GiB


In [2]:
# 2. Get a fresh copy of main
%cd /content
!rm -rf kv-evidence-loss
!git clone -q https://github.com/burakzor/kv-evidence-loss.git
%cd /content/kv-evidence-loss
!git log -1 --format="%h %ad %s" --date=short

/content
/content/kv-evidence-loss
401a542 2026-11-21 Add attention recorder and QRHead head detection


In [3]:
# 3. Install the pinned team environment (torch is Colab's own CUDA build)
!pip install -q -r requirements.txt
!pip list 2>/dev/null | grep -iE "^(torch|transformers|tokenizers|accelerate|datasets|pandas|numpy|pyarrow|tqdm) "

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
accelerate                            1.15.0
datasets                              4.8.5
numpy                                 2.4.6
pandas                                2.2.3
pyarrow                               23.0.1
tokenizers                            0.22.2
torch                                 2.14.1+cu130
tqdm                                  4.67.3
transformers                          5.2.0


In [4]:
# 4. Record the full environment of this run (GPU, driver, Python, CUDA, every package)
!(nvidia-smi --query-gpu=name,driver_version --format=csv,noheader; python --version; python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"; pip freeze) > data/hotpotqa/qrhead_detect_env.txt
!head -4 data/hotpotqa/qrhead_detect_env.txt

NVIDIA L4, 580.82.07
Python 3.13.16
torch 2.14.1+cu130 cuda 13.0
absl-py==2.0.0


In [5]:
# 5. Optional: Hugging Face token for a faster download (only works in the Colab web UI)
import os
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print(f"No HF token ({type(e).__name__}); downloading anonymously.")

No HF token (TimeoutException); downloading anonymously.


In [6]:
# 6. Detect the QRHead heads of Qwen3-8B on 128 bridge questions outside the clean set
# (first run downloads Qwen3-8B, ~16 GB; the detection itself takes about 10 minutes)
!python scripts/qrhead_detect.py

README.md: 100% 9.52k/9.52k [00:00<00:00, 2.45MB/s]

distractor/train-00000-of-00002.parquet: downloading bytes:   3% 4.78M/166M [00:01<00:54, 2.93MB/s]
distractor/train-00000-of-00002.parquet: downloading bytes:  20% 33.2M/166M [00:01<00:04, 28.3MB/s, 1.52MB/s  ]
distractor/train-00000-of-00002.parquet: downloading bytes:  46% 76.1M/166M [00:02<00:01, 65.6MB/s, 5.66MB/s  ]
distractor/train-00000-of-00002.parquet: downloading bytes:  72% 119M/166M [00:02<00:00, 78.7MB/s, 9.63MB/s  ]
distractor/train-00000-of-00002.parquet: downloading bytes:  93% 154M/166M [00:03<00:00, 97.6MB/s, 12.3MB/s  ]
distractor/train-00000-of-00002.parquet: downloading bytes: 100% 166M/166M [00:03<00:00, 46.9MB/s, 14.4MB/s  ]
distractor/train-00000-of-00002.parquet: reconstructing file: 100% 166M/166M [00:03<00:00, 46.9MB/s, 15.1MB/s  ]

distractor/train-00001-of-00002.parquet: downloading bytes:   3% 4.48M/166M [00:01<00:55, 2.94MB/s]
distractor/train-00001-of-00002.parquet: downloading bytes:  86% 143M/166M [

In [7]:
# 7. Look at the scores: the 16 best heads and the layers they are in
import json
import pandas as pd
scores = json.load(open("data/hotpotqa/qrhead_scores_qwen3_8b.json"))
scores_df = pd.DataFrame({"head": scores.keys(), "qrscore": scores.values()})
scores_df["layer"] = scores_df["head"].apply(lambda x: int(x.split("-")[0]))
top_df = scores_df.sort_values("qrscore", ascending=False).head(16)
print("heads scored:", len(scores_df))
print(top_df.to_string(index=False))
print(top_df["layer"].value_counts().sort_index())

heads scored: 1152
 head  qrscore  layer
23-10 0.435971     23
21-11 0.367874     21
20-15 0.358416     20
24-29 0.352352     24
24-13 0.350809     24
21-10 0.338065     21
24-23 0.334784     24
23-30 0.322952     23
18-15 0.321739     18
21-18 0.316235     21
 17-4 0.315983     17
 22-4 0.315501     22
 23-4 0.300828     23
 15-9 0.300066     15
 24-5 0.297925     24
18-19 0.294851     18
layer
15    1
17    1
18    2
20    1
21    3
22    1
23    3
24    4
Name: count, dtype: int64


In [8]:
# 8. Download the results and the environment record
from google.colab import files
files.download("data/hotpotqa/qrhead_scores_qwen3_8b.json")
files.download("data/hotpotqa/qrhead_heads_qwen3_8b.txt")
files.download("data/hotpotqa/qrhead_detection_ids.txt")
files.download("data/hotpotqa/qrhead_detect_env.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>